# Unsupervised scVI baseline

The unsupervised scVI counterpart of both HPO winners is trained with the same
architecture, schedule, and split: plain scVI for the joint approach, an SCVI
reference plus a `load_query_data` query for scArches. Both score below scANVI,
so semi-supervision contributes to matching.

Requires the processed objects, the saved HPO runs, and the fixed-k ranking
table. Artifacts go to `models/validation/scvi_baseline/`.

In [1]:
import json
import time
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import anndata as ad
import scvi

from src import hpo
from src import hpo_pp_utils as ppu
from src import model_io
from src.utils import load_config

CFG = load_config()
PATHS = CFG["paths"]
HPO_CFG = CFG["hpo"]

adata_tumor = ad.read_h5ad(ROOT / PATHS["processed"]["pp_tumor"])
adata_cl = ad.read_h5ad(ROOT / PATHS["processed"]["pp_cell_line"])
print(f"tumor {adata_tumor.shape}, cell line {adata_cl.shape}")

tumor (108131, 28617), cell line (2162, 28617)


# HPO winners

In [2]:
best = pd.read_csv(ROOT / "models" / "hpo" / "best_models_primary_100.csv")
best[["config_id", "num_hvg", "approach", "run_dir", "k100_mean_line_score"]]

,config_id,num_hvg,approach,run_dir,k100_mean_line_score
0,10,3000,joint,models/hpo/runs/hvg3000/config_10/joint,0.679064
1,47,10000,scarches,models/hpo/runs/hvg10000/config_47/scarches,0.519707


In [3]:
JOINT_RUN = ROOT / best.loc[best["approach"] == "joint", "run_dir"].iloc[0]
SCARCHES_RUN = ROOT / best.loc[best["approach"] == "scarches", "run_dir"].iloc[0]
joint_cfg = json.loads((JOINT_RUN / "config.json").read_text())
scarches_cfg = json.loads((SCARCHES_RUN / "config.json").read_text())
print(JOINT_RUN)
print(SCARCHES_RUN)

/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg3000/config_10/joint
/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg10000/config_47/scarches


# Joint winner: scVI baseline (hvg3000, config 10)

In [4]:
torch.set_float32_matmul_precision("high")
hpo.set_seed(joint_cfg["seed"])

combined_3k = model_io.build_combined_adata_for_hvg(
    adata_tumor, adata_cl, joint_cfg["num_hvg"], hvg_dir=JOINT_RUN.parents[1]
)
print(f"combined {combined_3k.shape}, "
      f"{combined_3k.obs['dataset'].value_counts().to_dict()}")

Seed set to 0


combined (110293, 3000), {'tumor': 108131, 'cell_line': 2162}


In [5]:
scvi_joint_dir = ROOT / "models" / "validation" / "scvi_baseline" / "joint_hvg3000"
scvi_model_path = scvi_joint_dir / "scvi"
scvi_joint_train_s = None
if scvi_model_path.exists():
    print(f"[load] {scvi_model_path}")
    scvi_joint = scvi.model.SCVI.load(str(scvi_model_path), adata=combined_3k)
else:
    scvi.model.SCVI.setup_anndata(
        combined_3k,
        layer="counts",
        batch_key=joint_cfg["batch_key"],
    )

    train_idx, val_idx = ppu.stratified_train_val_indices(
        combined_3k, joint_cfg["batch_key"], joint_cfg["labels_key"],
        seed=joint_cfg["seed"],
    )

    scvi_joint = scvi.model.SCVI(combined_3k, **hpo.scvi_model_kwargs(joint_cfg))
    t0 = time.time()
    scvi_joint.train(**hpo.training_kwargs(
        joint_cfg, joint_cfg["n_epochs_kl_warmup"]["joint"], [train_idx, val_idx]
    ))
    scvi_joint_train_s = time.time() - t0

    print(f"scVI joint baseline trained in {scvi_joint_train_s:.0f}s")
    scvi_joint.save(scvi_model_path, overwrite=True)

[load] /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/joint_hvg3000/scvi
INFO     File                                                                                                      
         /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/joint_hvg3000/scvi/model.pt    
         already downloaded                                                                                        


In [6]:
if hpo.is_run_done(scvi_joint_dir):
    print("[skip] evaluation already done")
else:
    hpo.evaluate(
        scvi_joint, combined_3k, joint_cfg, scvi_joint_dir, "joint",
        scib_n_cells=HPO_CFG["scib_n_cells"], train_time_s=scvi_joint_train_s,
    )
    hpo.mark_run_done(scvi_joint_dir)

[skip] evaluation already done


# scArches winner: scVI reference + query baseline (hvg10000, config 47)

In [7]:
hpo.set_seed(scarches_cfg["seed"])
hvg_dir_10k = SCARCHES_RUN.parents[1]
hvg_genes = (hvg_dir_10k / "hvg_genes.txt").read_text().splitlines()
tumor_hvg = adata_tumor[:, hvg_genes].copy()
cl_hvg = adata_cl[:, hvg_genes].copy()
tumor_hvg.layers["counts"] = tumor_hvg.X.copy()
cl_hvg.layers["counts"] = cl_hvg.X.copy()
combined_10k = ppu.build_combined_adata(tumor_hvg, cl_hvg)
print(f"combined {combined_10k.shape}")

Seed set to 0


combined (110293, 10000)


In [8]:
scvi_scarches_dir = ROOT / "models" / "validation" / "scvi_baseline" / "scarches_hvg10000"
if (scvi_scarches_dir / "reference").exists():
    print(f"[load] {scvi_scarches_dir / 'reference'}")
    scvi_ref = scvi.model.SCVI.load(str(scvi_scarches_dir / "reference"), adata=tumor_hvg)
else:
    scvi.model.SCVI.setup_anndata(
        tumor_hvg, layer="counts", batch_key=scarches_cfg["batch_key"],
    )
    ref_train_idx, ref_val_idx = ppu.stratified_train_val_indices(
        tumor_hvg, scarches_cfg["batch_key"], scarches_cfg["labels_key"],
        seed=scarches_cfg["seed"],
    )
    scvi_ref = scvi.model.SCVI(tumor_hvg, **hpo.scvi_model_kwargs(scarches_cfg))
    scvi_ref.train(**hpo.training_kwargs(
        scarches_cfg, scarches_cfg["n_epochs_kl_warmup"]["joint"],
        [ref_train_idx, ref_val_idx],
    ))
    scvi_ref.save(scvi_scarches_dir / "reference", overwrite=True)

[load] /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/scarches_hvg10000/reference
INFO     File                                                                                                      
         /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/scarches_hvg10000/reference/mod
         el.pt already downloaded                                                                                  


In [9]:
query_path = scvi_scarches_dir / "query"
if query_path.exists():
    print(f"[load] {query_path}")
    scvi_query = scvi.model.SCVI.load(str(query_path), adata=cl_hvg)
else:
    scvi_query = scvi.model.SCVI.load_query_data(cl_hvg, reference_model=scvi_ref)
    q_train_idx, q_val_idx = ppu.stratified_train_val_indices(
        cl_hvg, scarches_cfg["batch_key"], scarches_cfg["labels_key"],
        seed=scarches_cfg["seed"],
    )
    scvi_query.train(**hpo.training_kwargs(
        scarches_cfg, scarches_cfg["n_epochs_kl_warmup"]["scarches"],
        [q_train_idx, q_val_idx], load_best_on_end=False,
    ))
    scvi_query.save(query_path, overwrite=True)

[load] /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/scarches_hvg10000/query
INFO     File                                                                                                      
         /mnt/c/Users/marku/github/scMapping-public/models/validation/scvi_baseline/scarches_hvg10000/query/model.p
         t already downloaded                                                                                      


In [10]:
if hpo.is_run_done(scvi_scarches_dir):
    print("[skip] evaluation already done")
else:
    hpo.evaluate(
        scvi_query, combined_10k, scarches_cfg, scvi_scarches_dir, "scarches",
        scib_n_cells=HPO_CFG["scib_n_cells"],
    )
    hpo.mark_run_done(scvi_scarches_dir)

[skip] evaluation already done


# Comparison

In [11]:
from src.validation import line_table, summary_row


In [12]:
comparison = pd.DataFrame({
    "scANVI joint (HPO)": summary_row(JOINT_RUN),
    "scVI joint baseline": summary_row(scvi_joint_dir),
    "scANVI scArches (HPO)": summary_row(SCARCHES_RUN),
    "scVI scArches baseline": summary_row(scvi_scarches_dir),
}).T
comparison.round(4)

,mean_line_score,mean_mixing,mean_specificity,mean_coverage,mean_line_purity,scib_Total,scib_Batch_correction,scib_Bio_conservation,n_epochs
scANVI joint (HPO),0.6791,0.6923,0.9496,1.0000,0.5224,0.5929,0.5999,0.5883,75.0
scVI joint baseline,0.6430,0.6787,0.7947,1.0000,0.5317,0.5067,0.5721,0.4631,194.0
scANVI scArches (HPO),0.5197,0.3729,0.9468,0.9890,0.4442,0.5657,0.5649,0.5662,400.0
scVI scArches baseline,0.4625,0.3158,0.8367,0.9037,0.4202,0.4922,0.5313,0.4661,288.0


In [13]:
pd.DataFrame({
    "scANVI joint": line_table(JOINT_RUN)["line_score"],
    "scVI joint": line_table(scvi_joint_dir)["line_score"],
    "scANVI scArches": line_table(SCARCHES_RUN)["line_score"],
    "scVI scArches": line_table(scvi_scarches_dir)["line_score"],
}).round(4)

,scANVI joint,scVI joint,scANVI scArches,scVI scArches
cell_line,,,,
NCIH747,0.5159,0.5079,0.4360,0.4338
LS1034,0.7719,0.7346,0.6312,0.5866
RCM1,0.6536,0.6068,0.4706,0.4959
CCK81,0.6314,0.6101,0.7066,0.6228
SNUC4,0.7122,0.6825,0.5370,0.4503
CL34,0.6493,0.6127,0.5354,0.3916
LS180,0.7547,0.6385,0.4779,0.4001
OUMS23,0.6438,0.6598,0.3901,0.3914
HCC56,0.7049,0.6763,0.4506,0.4587
